
# MedQuAD → 500-token chunks → MedCPT embeddings → FAISS

This Google Colab notebook:

1. Downloads **MedQuAD** from Kaggle.
2. Uses the **answer/content** as the vector-store text.
3. Keeps the original question and other available fields as metadata.
4. Splits each answer into chunks of **maximum 500 tokens** with **50-token overlap**.
5. Embeds chunks with **`ncbi/MedCPT-Article-Encoder`**.
6. Builds a cosine-similarity **FAISS** index.
7. Saves the vector store to:

```text
/content/drive/MyDrive/vector store/
```

8. Tests retrieval with **`ncbi/MedCPT-Query-Encoder`** and displays the top-5 chunks.

> **Important:** MedCPT uses separate article and query encoders trained into the same embedding space. The vector store therefore uses the Article Encoder, while search queries use the Query Encoder.


## 1. Install dependencies

In [1]:
!pip -q install -U kagglehub transformers faiss-cpu pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.


## 2. Imports and configuration

In [2]:
import os
import json
import pickle
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import faiss
import kagglehub

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

# -----------------------------
# Configuration
# -----------------------------
KAGGLE_DATASET = "pythonafroz/medquad-medical-question-answer-for-ai-research"

ARTICLE_MODEL_NAME = "ncbi/MedCPT-Article-Encoder"
QUERY_MODEL_NAME   = "ncbi/MedCPT-Query-Encoder"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
BATCH_SIZE = 64

TOP_K = 5

DRIVE_DIR = Path("/content/drive/MyDrive/vector store")
INDEX_PATH = DRIVE_DIR / "medquad_medcpt.faiss"
METADATA_PATH = DRIVE_DIR / "medquad_metadata.pkl"
CONFIG_PATH = DRIVE_DIR / "medquad_vectorstore_config.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


## 3. Mount Google Drive

In [3]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print("Vector store will be saved to:")
print(DRIVE_DIR)

Mounted at /content/drive
Vector store will be saved to:
/content/drive/MyDrive/vector store



## 4. Download MedQuAD from Kaggle

This uses the public Kaggle dataset:

`pythonafroz/medquad-medical-question-answer-for-ai-research`

For a public dataset, `kagglehub` normally downloads it directly. If Kaggle asks for authentication in your environment, authenticate your Kaggle account and rerun this cell.


In [4]:
dataset_dir = Path(kagglehub.dataset_download(KAGGLE_DATASET))

print("Downloaded dataset directory:")
print(dataset_dir)

all_files = [p for p in dataset_dir.rglob("*") if p.is_file()]
print("\nFiles found:")
for p in all_files[:30]:
    print(" -", p.relative_to(dataset_dir))

if len(all_files) > 30:
    print(f"... and {len(all_files) - 30} more files")

100%|██████████| 4.95M/4.95M [00:00<00:00, 53.4MB/s]

Extracting files...


Downloaded dataset directory:
/root/.cache/kagglehub/datasets/pythonafroz/medquad-medical-question-answer-for-ai-research/versions/1

Files found:
 - medquad.csv


## 5. Locate and load the MedQuAD table

In [5]:
csv_files = list(dataset_dir.rglob("*.csv"))

if not csv_files:
    raise FileNotFoundError(
        "No CSV file was found in the downloaded Kaggle dataset. "
        "Inspect the files printed above and adjust the loader if Kaggle changes the package format."
    )

preferred = [p for p in csv_files if "medquad" in p.name.lower()]
csv_path = preferred[0] if preferred else csv_files[0]

print("Using CSV:", csv_path)

df = pd.read_csv(csv_path)
print("Shape:", df.shape)
print("Columns:", list(df.columns))

display(df.head(3))

Using CSV: /root/.cache/kagglehub/datasets/pythonafroz/medquad-medical-question-answer-for-ai-research/versions/1/medquad.csv
Shape: (16412, 4)
Columns: ['question', 'answer', 'source', 'focus_area']


,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma



## 6. Normalize column names and clean the data

The Kaggle MedQuAD CSV is commonly distributed with columns such as:

- `question`
- `answer`
- `source`
- `focus_area`

The code below is tolerant of capitalization and minor naming differences. Only the **answer/content** is embedded. Other available fields are preserved as metadata.


In [6]:
normalized_to_original = {
    str(col).strip().lower().replace(" ", "_"): col
    for col in df.columns
}

def find_column(candidates):
    for candidate in candidates:
        key = candidate.strip().lower().replace(" ", "_")
        if key in normalized_to_original:
            return normalized_to_original[key]
    return None

ANSWER_COL = find_column(["answer", "answers", "response", "content"])
QUESTION_COL = find_column(["question", "questions", "query"])
SOURCE_COL = find_column(["source", "url", "source_url"])
FOCUS_COL = find_column(["focus_area", "focus", "topic", "category"])
QTYPE_COL = find_column(["qtype", "question_type", "type"])

if ANSWER_COL is None:
    raise ValueError(
        f"Could not identify an answer/content column. Available columns: {list(df.columns)}"
    )

print("Detected columns:")
print("  Answer :", ANSWER_COL)
print("  Question:", QUESTION_COL)
print("  Source  :", SOURCE_COL)
print("  Focus   :", FOCUS_COL)
print("  Q-type  :", QTYPE_COL)

# Remove rows with missing/empty answer text.
work_df = df.copy()
work_df = work_df[work_df[ANSWER_COL].notna()].copy()
work_df[ANSWER_COL] = work_df[ANSWER_COL].astype(str).str.strip()
work_df = work_df[work_df[ANSWER_COL].str.len() > 0].copy()
work_df = work_df.drop_duplicates(subset=[ANSWER_COL]).reset_index(drop=True)

print("\nUsable unique answers:", len(work_df))

Detected columns:
  Answer : answer
  Question: question
  Source  : source
  Focus   : focus_area
  Q-type  : None

Usable unique answers: 15817


## 7. Load MedCPT Article Encoder

In [7]:
article_tokenizer = AutoTokenizer.from_pretrained(ARTICLE_MODEL_NAME)
article_model = AutoModel.from_pretrained(ARTICLE_MODEL_NAME)
article_model = article_model.to(DEVICE)
article_model.eval()

print("Loaded:", ARTICLE_MODEL_NAME)
print("Embedding dimension:", article_model.config.hidden_size)
print("Model maximum positions:", article_model.config.max_position_embeddings)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/706k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded: ncbi/MedCPT-Article-Encoder
Embedding dimension: 768
Model maximum positions: 512



## 8. Token-aware chunking: 500 tokens with 50-token overlap

Chunking uses the **MedCPT Article Encoder tokenizer itself**, so the 500-token limit is measured in the tokenizer actually used for embedding.

The chunks contain only the answer/content. The original question, source, focus area, and other fields are stored as metadata but are **not concatenated into the embedded text**.


In [8]:
def chunk_text_by_tokens(text, tokenizer, chunk_size=500, overlap=50):
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    token_ids = tokenizer.encode(
        str(text),
        add_special_tokens=False,
        truncation=False,
    )

    if not token_ids:
        return []

    step = chunk_size - overlap
    chunks = []

    for start in range(0, len(token_ids), step):
        end = min(start + chunk_size, len(token_ids))
        chunk_ids = token_ids[start:end]

        chunk_text = tokenizer.decode(
            chunk_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        ).strip()

        if chunk_text:
            chunks.append({
                "text": chunk_text,
                "token_count": len(chunk_ids),
                "start_token": start,
                "end_token": end,
            })

        if end >= len(token_ids):
            break

    return chunks


# Quick sanity check
sample_text = work_df.iloc[0][ANSWER_COL]
sample_chunks = chunk_text_by_tokens(
    sample_text,
    article_tokenizer,
    CHUNK_SIZE,
    CHUNK_OVERLAP,
)

print("Sample answer token chunks:", len(sample_chunks))
for i, c in enumerate(sample_chunks[:3]):
    print(f"Chunk {i}: {c['token_count']} tokens")

Sample answer token chunks: 1
Chunk 0: 395 tokens


## 9. Create chunks and metadata

In [9]:
chunk_records = []

for row_id, row in tqdm(
    work_df.iterrows(),
    total=len(work_df),
    desc="Chunking MedQuAD answers"
):
    answer = str(row[ANSWER_COL])

    chunks = chunk_text_by_tokens(
        answer,
        article_tokenizer,
        chunk_size=CHUNK_SIZE,
        overlap=CHUNK_OVERLAP,
    )

    for chunk_id, chunk in enumerate(chunks):
        record = {
            "row_id": int(row_id),
            "chunk_id": int(chunk_id),
            "text": chunk["text"],
            "token_count": int(chunk["token_count"]),
            "start_token": int(chunk["start_token"]),
            "end_token": int(chunk["end_token"]),
        }

        # Preserve useful metadata when available.
        if QUESTION_COL is not None:
            record["question"] = (
                None if pd.isna(row[QUESTION_COL]) else str(row[QUESTION_COL])
            )
        if SOURCE_COL is not None:
            record["source"] = (
                None if pd.isna(row[SOURCE_COL]) else str(row[SOURCE_COL])
            )
        if FOCUS_COL is not None:
            record["focus_area"] = (
                None if pd.isna(row[FOCUS_COL]) else str(row[FOCUS_COL])
            )
        if QTYPE_COL is not None:
            record["question_type"] = (
                None if pd.isna(row[QTYPE_COL]) else str(row[QTYPE_COL])
            )

        chunk_records.append(record)

print("Original usable answers:", len(work_df))
print("Total chunks:", len(chunk_records))

if not chunk_records:
    raise RuntimeError("No chunks were created.")

print("\nFirst metadata record:")
print(json.dumps(chunk_records[0], indent=2, ensure_ascii=False)[:2000])

Chunking MedQuAD answers:   0%|          | 0/15817 [00:00<?, ?it/s]

Original usable answers: 15817
Total chunks: 18142

First metadata record:
{
  "row_id": 0,
  "chunk_id": 0,
  "text": "glaucoma is a group of diseases that can damage the eye's optic nerve and result in vision loss and blindness. while glaucoma can strike anyone, the risk is much greater for people over 60. how glaucoma develops there are several different types of glaucoma. most of these involve the drainage system within the eye. at the front of the eye there is a small space called the anterior chamber. a clear fluid flows through this chamber and bathes and nourishes the nearby tissues. ( watch the video to learn more about glaucoma. to enlarge the video, click the brackets in the lower right - hand corner. to reduce the video, press the escape ( esc ) button on your keyboard. ) in glaucoma, for still unknown reasons, the fluid drains too slowly out of the eye. as the fluid builds up, the pressure inside the eye rises. unless this pressure is controlled, it may cause damage to the


## 10. Embed answer chunks with MedCPT Article Encoder

MedCPT's article encoder normally accepts `(title, abstract)` pairs. Because you requested **answer/content-only embedding**, each chunk is passed as the article body with a blank title.

The `[CLS]` representation is used, following the MedCPT model card.


In [10]:
@torch.inference_mode()
def embed_article_chunks(texts, batch_size=64):
    all_embeddings = []

    for start in tqdm(
        range(0, len(texts), batch_size),
        desc="Embedding chunks"
    ):
        batch_texts = texts[start:start + batch_size]

        # Blank title + answer chunk as body.
        article_pairs = [["", text] for text in batch_texts]

        encoded = article_tokenizer(
            article_pairs,
            truncation=True,
            padding=True,
            max_length=512,
            return_tensors="pt",
        )
        encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

        outputs = article_model(**encoded)
        embeddings = outputs.last_hidden_state[:, 0, :]

        embeddings = embeddings.float().cpu().numpy().astype("float32")
        all_embeddings.append(embeddings)

    return np.vstack(all_embeddings)


chunk_texts = [record["text"] for record in chunk_records]

try:
    embeddings = embed_article_chunks(chunk_texts, batch_size=BATCH_SIZE)
except torch.cuda.OutOfMemoryError:
    # Automatic fallback for smaller Colab GPUs.
    print("CUDA OOM at batch size", BATCH_SIZE, "— retrying with batch size 32.")
    torch.cuda.empty_cache()
    gc.collect()
    embeddings = embed_article_chunks(chunk_texts, batch_size=32)

print("Embedding matrix shape:", embeddings.shape)

Embedding chunks:   0%|          | 0/284 [00:00<?, ?it/s]

Embedding matrix shape: (18142, 768)



## 11. Build FAISS cosine-similarity index

FAISS `IndexFlatIP` performs inner-product search. After L2-normalizing document and query embeddings, inner product is equivalent to cosine similarity.


In [11]:
faiss.normalize_L2(embeddings)

embedding_dim = embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings)

print("FAISS index type:", type(index).__name__)
print("Vectors indexed:", index.ntotal)
print("Embedding dimension:", embedding_dim)

FAISS index type: IndexFlatIP
Vectors indexed: 18142
Embedding dimension: 768


## 12. Save FAISS vector store to Google Drive

In [12]:
faiss.write_index(index, str(INDEX_PATH))

# This metadata file is part of the vector store and maps each FAISS vector
# back to its chunk text and original MedQuAD metadata.
with open(METADATA_PATH, "wb") as f:
    pickle.dump(chunk_records, f, protocol=pickle.HIGHEST_PROTOCOL)

config = {
    "dataset": KAGGLE_DATASET,
    "answer_column": str(ANSWER_COL),
    "question_column": None if QUESTION_COL is None else str(QUESTION_COL),
    "source_column": None if SOURCE_COL is None else str(SOURCE_COL),
    "focus_column": None if FOCUS_COL is None else str(FOCUS_COL),
    "question_type_column": None if QTYPE_COL is None else str(QTYPE_COL),
    "article_embedding_model": ARTICLE_MODEL_NAME,
    "query_embedding_model": QUERY_MODEL_NAME,
    "chunk_size_tokens": CHUNK_SIZE,
    "chunk_overlap_tokens": CHUNK_OVERLAP,
    "distance": "cosine similarity via normalized IndexFlatIP",
    "embedding_dimension": int(embedding_dim),
    "number_of_vectors": int(index.ntotal),
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("Saved:")
print(" -", INDEX_PATH)
print(" -", METADATA_PATH)
print(" -", CONFIG_PATH)

print("\nFile sizes:")
for p in [INDEX_PATH, METADATA_PATH, CONFIG_PATH]:
    print(f"{p.name}: {p.stat().st_size / (1024**2):.2f} MB")

Saved:
 - /content/drive/MyDrive/vector store/medquad_medcpt.faiss
 - /content/drive/MyDrive/vector store/medquad_metadata.pkl
 - /content/drive/MyDrive/vector store/medquad_vectorstore_config.json

File sizes:
medquad_medcpt.faiss: 53.15 MB
medquad_metadata.pkl: 22.44 MB
medquad_vectorstore_config.json: 0.00 MB



## 13. Free the Article Encoder before retrieval testing

The vector store is already built, so the article model is no longer needed for query-time retrieval.


In [13]:
del article_model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Article encoder released.")

Article encoder released.


## 14. Load MedCPT Query Encoder

In [14]:
query_tokenizer = AutoTokenizer.from_pretrained(QUERY_MODEL_NAME)
query_model = AutoModel.from_pretrained(QUERY_MODEL_NAME)
query_model = query_model.to(DEVICE)
query_model.eval()

print("Loaded:", QUERY_MODEL_NAME)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/706k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded: ncbi/MedCPT-Query-Encoder


## 15. Reload the saved vector store from Google Drive

In [15]:
loaded_index = faiss.read_index(str(INDEX_PATH))

with open(METADATA_PATH, "rb") as f:
    loaded_metadata = pickle.load(f)

print("Loaded FAISS vectors:", loaded_index.ntotal)
print("Loaded metadata records:", len(loaded_metadata))

assert loaded_index.ntotal == len(loaded_metadata), (
    "FAISS vector count and metadata count do not match."
)

Loaded FAISS vectors: 18142
Loaded metadata records: 18142


## 16. Retrieval function

In [16]:
@torch.inference_mode()
def embed_query(query):
    encoded = query_tokenizer(
        [query],
        truncation=True,
        padding=True,
        max_length=64,
        return_tensors="pt",
    )
    encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

    outputs = query_model(**encoded)
    query_embedding = outputs.last_hidden_state[:, 0, :]
    query_embedding = query_embedding.float().cpu().numpy().astype("float32")

    faiss.normalize_L2(query_embedding)
    return query_embedding


def retrieve(query, top_k=5):
    query_embedding = embed_query(query)
    scores, indices = loaded_index.search(query_embedding, top_k)

    results = []
    for rank, (idx, score) in enumerate(
        zip(indices[0], scores[0]),
        start=1
    ):
        if idx < 0:
            continue

        item = dict(loaded_metadata[idx])
        item["rank"] = rank
        item["score"] = float(score)
        item["faiss_id"] = int(idx)
        results.append(item)

    return results

## 17. Top-5 retrieval test

In [17]:
query = "What are the symptoms of diabetes?"

results = retrieve(query, top_k=TOP_K)

print("QUERY:", query)
print("=" * 100)

for result in results:
    print(f"Rank: {result['rank']} | Cosine score: {result['score']:.4f}")

    if result.get("question"):
        print("Original MedQuAD question:", result["question"])

    if result.get("focus_area"):
        print("Focus area:", result["focus_area"])

    if result.get("question_type"):
        print("Question type:", result["question_type"])

    if result.get("source"):
        print("Source:", result["source"])

    print(
        f"Chunk: row={result['row_id']}, "
        f"chunk={result['chunk_id']}, "
        f"tokens={result['token_count']}"
    )
    print("Text:")
    print(result["text"])
    print("-" * 100)

QUERY: What are the symptoms of diabetes?
Rank: 1 | Cosine score: 0.7458
Original MedQuAD question: What is the outlook for Diabetic Neuropathy ?
Focus area: Diabetic Neuropathy
Source: NINDS
Chunk: row=8940, chunk=0, tokens=53
Text:
the prognosis for diabetic neuropathy depends largely on how well the underlying condition of diabetes is handled. treating diabetes may halt progression and improve symptoms of the neuropathy, but recovery is slow. the painful sensations of diabetic neuropathy may become severe enough to cause depression in some patients.
----------------------------------------------------------------------------------------------------
Rank: 2 | Cosine score: 0.7412
Original MedQuAD question: How many people are affected by nephrogenic diabetes insipidus ?
Focus area: nephrogenic diabetes insipidus
Source: GHR
Chunk: row=12867, chunk=0, tokens=32
Text:
the prevalence of nephrogenic diabetes insipidus is unknown, although the condition is thought to be rare. the acquired


## Saved vector-store structure

After completion, Google Drive will contain:

```text
MyDrive/
└── vector store/
    ├── medquad_medcpt.faiss
    ├── medquad_metadata.pkl
    └── medquad_vectorstore_config.json
```

- `medquad_medcpt.faiss` — normalized MedCPT chunk embeddings.
- `medquad_metadata.pkl` — chunk text plus associated MedQuAD metadata.
- `medquad_vectorstore_config.json` — reproducibility/configuration information.

No separate processed/chunked dataset is saved.
